<div align="center">

  <a href="https://www.ultralytics.com/yolo/yolo27?utm_source=github&utm_medium=social&utm_campaign=yolo27-launch-2026&utm_content=banner" target="_blank">
    <img width="1024", src="https://raw.githubusercontent.com/ultralytics/assets/main/yolov8/banner-yolov8.png"></a>

  [中文](https://docs.ultralytics.com/zh) | [한국어](https://docs.ultralytics.com/ko) | [日本語](https://docs.ultralytics.com/ja) | [Русский](https://docs.ultralytics.com/ru) | [Deutsch](https://docs.ultralytics.com/de) | [Français](https://docs.ultralytics.com/fr) | [Español](https://docs.ultralytics.com/es) | [Português](https://docs.ultralytics.com/pt) | [Türkçe](https://docs.ultralytics.com/tr) | [Tiếng Việt](https://docs.ultralytics.com/vi) | [العربية](https://docs.ultralytics.com/ar)

  <a href="https://github.com/ultralytics/ultralytics/actions/workflows/ci.yml"><img src="https://github.com/ultralytics/ultralytics/actions/workflows/ci.yml/badge.svg" alt="Ultralytics CI"></a>
  <a href="https://clickpy.clickhouse.com/dashboard/ultralytics"><img src="https://static.pepy.tech/badge/ultralytics" alt="Ultralytics Downloads"></a>
  <a href="https://discord.com/invite/ultralytics"><img alt="Ultralytics Discord" src="https://img.shields.io/discord/1089800235347353640?logo=discord&logoColor=white&label=Discord&color=blue"></a>
  <a href="https://community.ultralytics.com"><img alt="Ultralytics Forums" src="https://img.shields.io/discourse/users?server=https%3A%2F%2Fcommunity.ultralytics.com&logo=discourse&label=Forums&color=blue"></a>
  <a href="https://www.reddit.com/r/ultralytics/"><img alt="Ultralytics Reddit" src="https://img.shields.io/reddit/subreddit-subscribers/ultralytics?style=flat&logo=reddit&logoColor=white&label=Reddit&color=blue"></a>
  <br>
  <a href="https://console.paperspace.com/github/ultralytics/notebooks/blob/main/notebooks/how-to-use-ultralytics-yolo-high-resolution-tiling.ipynb"><img src="https://assets.paperspace.io/img/gradient-badge.svg" alt="Run Ultralytics on Gradient"></a>
  <a href="https://colab.research.google.com/github/ultralytics/notebooks/blob/main/notebooks/how-to-use-ultralytics-yolo-high-resolution-tiling.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open Ultralytics In Colab"></a>
  <a href="https://www.kaggle.com/models/ultralytics/yolo26"><img src="https://kaggle.com/static/images/open-in-kaggle.svg" alt="Open Ultralytics In Kaggle"></a>
  <a href="https://mybinder.org/v2/gh/ultralytics/notebooks/HEAD?labpath=notebooks%2Fhow-to-use-ultralytics-yolo-high-resolution-tiling.ipynb"><img src="https://mybinder.org/badge_logo.svg" alt="Open Ultralytics In Binder"></a>

  Welcome to the <a href="https://docs.ultralytics.com/guides/high-resolution-tiling">Ultralytics YOLO26 High Resolution Tiling notebook 🚀</a>. YOLO26 is the latest version of the YOLO (You Only Look Once) AI models developed by <a href="https://www.ultralytics.com">Ultralytics</a>. We hope that the resources in this notebook will help you detect small objects in large images with YOLO26. Please browse the YOLO26 <a href="https://docs.ultralytics.com">Docs</a> for details, raise an issue on <a href="https://github.com/ultralytics/ultralytics">GitHub</a> for support, and join our <a href="https://ultralytics.com/discord">Discord</a> community for questions and discussions!</div>

# High Resolution Tiling with Ultralytics YOLO26

This notebook shows how to detect small objects in large images and videos with the Ultralytics `tile` argument. See the [High Resolution Tiling guide](https://docs.ultralytics.com/guides/high-resolution-tiling) for every task and option.

### What is tiling?

YOLO26 resizes each image so its longest side matches `imgsz` (640 by default), so small objects in a large image can shrink to a few pixels. With `tile=True`, Ultralytics splits each image into overlapping windows, runs the model on every window, and merges the detections back into one result per original image.

- **Built in:** `tile` works in `predict`, `track`, `val`, and `train`, with no extra package.
- **Simple sizing:** `tile=True` uses `imgsz` as the window size, and an integer such as `tile=1024` sets it separately. Windows overlap by 20%. A window smaller than `imgsz`, such as `tile=400`, is enlarged to the model input, which helps when objects are only a few pixels wide. Much smaller windows can also enlarge background clutter into false positives.
- **Trade-off:** every window is another model pass, and each window sees less of the scene, so compare tiled and whole-image results on your own data.

### Setup

pip install `ultralytics` and [dependencies](https://github.com/ultralytics/ultralytics/blob/main/pyproject.toml) and check software and hardware.

[![PyPI - Version](https://img.shields.io/pypi/v/ultralytics?logo=pypi&logoColor=white)](https://pypi.org/project/ultralytics/) [![Downloads](https://static.pepy.tech/badge/ultralytics)](https://clickpy.clickhouse.com/dashboard/ultralytics) [![PyPI - Python Version](https://img.shields.io/pypi/pyversions/ultralytics?logo=python&logoColor=gold)](https://pypi.org/project/ultralytics/)

In [ ]:
!uv pip install ultralytics
import ultralytics
ultralytics.checks()

### Download the Sample Video

This 960 × 1080 aerial video of a roundabout is full of small cars. If you want to use your own image or video, you can skip this step.

In [ ]:
from ultralytics.utils.downloads import safe_download

safe_download("https://github.com/ultralytics/assets/releases/download/v0.0.0/sahi.demo.video.mp4", file="roundabout.mp4")

### Compare Whole-Image and Tiled Prediction

This frame is only 960 × 1080, so `tile=True` would cut 640-pixel windows that show the cars at their native size, barely larger than in the whole-image pass. With `tile=400`, each 400-pixel window is enlarged 1.6× to the 640-pixel model input, so the small parked cars become large enough to detect. Run YOLO26s on the first frame twice: once on the whole frame and once with `tile=400`.

In [ ]:
import cv2

from ultralytics import YOLO

model = YOLO("yolo26s.pt")

cap = cv2.VideoCapture("roundabout.mp4")
_, frame = cap.read()
cap.release()

whole = model.predict(frame, verbose=False)[0]
tiled = model.predict(frame, tile=400, verbose=False)[0]
print(f"Whole image: {len(whole.boxes)} detections, tiled: {len(tiled.boxes)} detections")
tiled.show()

<p align="center">
  <img width="1024" src="https://cdn.ul.run/i/d6e96fdf102af52175bee1612771b4df.avif" alt="YOLO26s on an aerial roundabout frame: 45 detections on the whole image (left) and 102 with tile=400 (right)">
</p>

On this frame, YOLO26s returns 45 detections on the whole image and 102 with `tile=400`, including 31 cars in the top-left parking lot instead of 6.

### Tiled Video Prediction

Use `stream=True` to process the video frame by frame with bounded memory. `save=True` writes the annotated video to `runs/detect/predict`, and `batch` sets how many windows run together. With `tile=400`, each frame runs as 12 windows, so use a GPU runtime for the full video.

In [ ]:
counts = [len(r.boxes) for r in model.predict("roundabout.mp4", tile=400, stream=True, save=True, verbose=False)]
print(f"{len(counts)} frames, {sum(counts) / len(counts):.1f} detections per frame on average")

The same options work from the command line:

```bash
yolo predict model=yolo26s.pt source=roundabout.mp4 tile=400 save=True
```

To follow each car across frames, use `model.track("roundabout.mp4", tile=400)`. The tracker runs once per frame on the merged detections.

### Train and Validate with Tiles

Use the same `tile` setting for training and validation when your images are much larger than `imgsz`. Training cuts each image into windows, clips labels at window edges, and keeps empty windows as negative examples. Validation merges tile predictions before scoring them against the original annotations.

```python
from ultralytics import YOLO

model = YOLO("yolo26n.pt")
model.train(data="VisDrone.yaml", tile=True, imgsz=640, epochs=100)
metrics = model.val(data="VisDrone.yaml", tile=True, imgsz=640)
```

The VisDrone dataset downloads about 2 GB, so this example is not run here. See the [High Resolution Tiling guide](https://docs.ultralytics.com/guides/high-resolution-tiling) for supported tasks, memory, and quality notes.